# 🎭 Ensemble Learning + Advanced Topics
---
Covers: Voting, Stacking, Bagging, Boosting, SHAP,
Feature Importance, Learning Curves, Model Explainability,
and all remaining advanced interview topics.

In [ ]:
# ============================================================================
# SETUP
# ============================================================================
import pandas as pd, numpy as np, warnings
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import (train_test_split, cross_val_score,
    StratifiedKFold, learning_curve)
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier,
    VotingClassifier, StackingClassifier, BaggingClassifier, AdaBoostClassifier)
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
    recall_score, roc_auc_score, confusion_matrix, classification_report)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from imblearn.over_sampling import SMOTE
warnings.filterwarnings('ignore')

# Load & Quick Clean
df = pd.read_csv('loan_default_dataset.csv')
for c in df.select_dtypes(include=[np.number]).columns:
    df[c].fillna(df[c].median(), inplace=True)
for c in df.select_dtypes(include=['object']).columns:
    df[c].fillna(df[c].mode()[0], inplace=True)
    if df[c].nunique() == 2:
        df[c] = LabelEncoder().fit_transform(df[c])
    else:
        df = pd.concat([df.drop(c, axis=1),
                        pd.get_dummies(df[c], prefix=c, drop_first=True)], axis=1)

X = df.drop('Default', axis=1)
y = df['Default']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_s = pd.DataFrame(scaler.fit_transform(X_train),
                          columns=X_train.columns, index=X_train.index)
X_test_s = pd.DataFrame(scaler.transform(X_test),
                         columns=X_test.columns, index=X_test.index)
smote = SMOTE(random_state=42)
X_bal, y_bal = smote.fit_resample(X_train_s, y_train)
print(f'✅ Data ready! Train: {len(X_bal)}, Test: {len(X_test)}')

---
## 🎭 PART 1: ENSEMBLE LEARNING
---

**Analogy (Like You're 10):**
- **Single model** = Asking ONE friend for advice
- **Ensemble** = Asking 10 friends and going with the majority!
- The "wisdom of the crowd" is usually better than any single expert.

### Three Types:
1. **Bagging** (Bootstrap Aggregating) → Parallel, reduces VARIANCE
2. **Boosting** → Sequential, reduces BIAS
3. **Stacking** → Combine different models with a meta-learner

In [ ]:
# ============================================================================
# 1. BAGGING — Bootstrap Aggregating
# ============================================================================
# WHAT: Train many models on RANDOM SUBSETS of data (with replacement)
#       Each model is independent. Combine by voting/averaging.
# WHY: Reduces VARIANCE (overfitting). Like asking many independent experts.
# EXAMPLE: Random Forest is bagging of Decision Trees!

print('1️⃣ BAGGING ENSEMBLE')
print('=' * 60)

# Bagging with Decision Trees
bag_dt = BaggingClassifier(
    estimator=DecisionTreeClassifier(max_depth=5),
    n_estimators=100,
    max_samples=0.8,      # Each tree sees 80% of data
    max_features=0.8,     # Each tree uses 80% of features
    random_state=42,
    n_jobs=-1
)
bag_dt.fit(X_bal, y_bal)
y_pred_bag = bag_dt.predict(X_test_s)
print(f'Bagging (DT) → F1: {f1_score(y_test, y_pred_bag):.4f}, '
      f'AUC: {roc_auc_score(y_test, bag_dt.predict_proba(X_test_s)[:,1]):.4f}')

# Random Forest IS bagging!
rf = RandomForestClassifier(n_estimators=200, max_depth=10,
                             class_weight='balanced', random_state=42, n_jobs=-1)
rf.fit(X_bal, y_bal)
y_pred_rf = rf.predict(X_test_s)
print(f'Random Forest → F1: {f1_score(y_test, y_pred_rf):.4f}, '
      f'AUC: {roc_auc_score(y_test, rf.predict_proba(X_test_s)[:,1]):.4f}')

print('\n💡 Random Forest = Bagging + Random Feature Selection')
print('   Each tree: random data subset + random feature subset → majority vote')

In [ ]:
# ============================================================================
# 2. BOOSTING — Sequential Error Correction
# ============================================================================
# WHAT: Train models SEQUENTIALLY. Each new model focuses on
#       the MISTAKES of previous models.
# WHY: Reduces BIAS (underfitting). Like a tutor fixing student's errors.
# EXAMPLES: AdaBoost, Gradient Boosting, XGBoost, LightGBM

print('2️⃣ BOOSTING ENSEMBLE')
print('=' * 60)

boosting_models = {
    'AdaBoost': AdaBoostClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=200, learning_rate=0.1, max_depth=5, random_state=42),
    'XGBoost': XGBClassifier(
        n_estimators=200, learning_rate=0.1, max_depth=5,
        random_state=42, use_label_encoder=False,
        eval_metric='logloss', verbosity=0),
    'LightGBM': LGBMClassifier(
        n_estimators=200, learning_rate=0.1, max_depth=5,
        random_state=42, verbose=-1),
}

for name, model in boosting_models.items():
    model.fit(X_bal, y_bal)
    yp = model.predict(X_test_s)
    yprob = model.predict_proba(X_test_s)[:, 1]
    print(f'{name:20s} → F1: {f1_score(y_test, yp):.4f}, '
          f'AUC: {roc_auc_score(y_test, yprob):.4f}')

print('\n💡 Boosting Progression:')
print('  AdaBoost → Gradient Boosting → XGBoost → LightGBM')
print('  Each is an improvement over the previous!')
print('  XGBoost: +Regularization, +Speed, +Missing handling')
print('  LightGBM: +Leaf-wise growth, +Fastest')

In [ ]:
# ============================================================================
# 3. VOTING CLASSIFIER — Combine Different Models
# ============================================================================
# WHAT: Train DIFFERENT types of models. Combine their predictions.
# Hard Voting: Majority vote (like election)
# Soft Voting: Average probabilities (better!)

print('3️⃣ VOTING CLASSIFIER')
print('=' * 60)

voting_clf = VotingClassifier(
    estimators=[
        ('lr', LogisticRegression(max_iter=1000, class_weight='balanced',
                                   random_state=42)),
        ('rf', RandomForestClassifier(n_estimators=100, max_depth=10,
                                       class_weight='balanced', random_state=42)),
        ('xgb', XGBClassifier(n_estimators=100, random_state=42,
                               use_label_encoder=False,
                               eval_metric='logloss', verbosity=0)),
        ('lgbm', LGBMClassifier(n_estimators=100, random_state=42, verbose=-1)),
    ],
    voting='soft'  # Use probabilities (better than hard voting)
)

voting_clf.fit(X_bal, y_bal)
yp = voting_clf.predict(X_test_s)
yprob = voting_clf.predict_proba(X_test_s)[:, 1]
print(f'Voting (Soft) → F1: {f1_score(y_test, yp):.4f}, '
      f'AUC: {roc_auc_score(y_test, yprob):.4f}')

print('\n💡 Why Voting Works:')
print('  Different models make DIFFERENT mistakes.')
print('  LR catches linear patterns, RF catches non-linear,')
print('  XGB catches complex interactions.')
print('  Combining them covers more ground!')

In [ ]:
# ============================================================================
# 4. STACKING CLASSIFIER — Meta-Learner
# ============================================================================
# WHAT: Train base models → Use their predictions as INPUT
#       to a FINAL model (meta-learner).
# Analogy: 3 doctors give opinions → 1 senior doctor makes final call.

print('4️⃣ STACKING CLASSIFIER')
print('=' * 60)

stacking_clf = StackingClassifier(
    estimators=[
        ('lr', LogisticRegression(max_iter=1000, random_state=42)),
        ('rf', RandomForestClassifier(n_estimators=100, random_state=42)),
        ('xgb', XGBClassifier(n_estimators=100, random_state=42,
                               use_label_encoder=False,
                               eval_metric='logloss', verbosity=0)),
    ],
    final_estimator=LogisticRegression(),  # Meta-learner
    cv=5,  # Use CV predictions to train meta-learner
    n_jobs=-1
)

stacking_clf.fit(X_bal, y_bal)
yp = stacking_clf.predict(X_test_s)
yprob = stacking_clf.predict_proba(X_test_s)[:, 1]
print(f'Stacking → F1: {f1_score(y_test, yp):.4f}, '
      f'AUC: {roc_auc_score(y_test, yprob):.4f}')

print('\n💡 Stacking Architecture:')
print('  Layer 1 (Base): LR, RF, XGB → each makes predictions')
print('  Layer 2 (Meta): LR learns to combine Layer 1 predictions')
print('  The meta-learner learns WHICH base model to trust more!')

In [ ]:
# ============================================================================
# ENSEMBLE COMPARISON
# ============================================================================
print('📊 ALL ENSEMBLE METHODS COMPARISON')
print('=' * 60)

ensemble_results = []
all_models = {
    'Single DT': DecisionTreeClassifier(max_depth=5, random_state=42),
    'Bagging (DT)': bag_dt,
    'Random Forest': rf,
    'AdaBoost': boosting_models['AdaBoost'],
    'Gradient Boosting': boosting_models['Gradient Boosting'],
    'XGBoost': boosting_models['XGBoost'],
    'LightGBM': boosting_models['LightGBM'],
    'Voting': voting_clf,
    'Stacking': stacking_clf,
}

for name, model in all_models.items():
    yp = model.predict(X_test_s)
    try:
        yprob = model.predict_proba(X_test_s)[:, 1]
        auc = roc_auc_score(y_test, yprob)
    except:
        auc = 0
    ensemble_results.append({
        'Model': name,
        'F1': round(f1_score(y_test, yp), 4),
        'AUC': round(auc, 4),
        'Accuracy': round(accuracy_score(y_test, yp), 4)
    })

res_df = pd.DataFrame(ensemble_results).sort_values('F1', ascending=False)
print(res_df.to_string(index=False))

# Visualize
fig, ax = plt.subplots(figsize=(10, 6))
sorted_df = res_df.sort_values('F1')
bars = ax.barh(sorted_df['Model'], sorted_df['F1'],
               color=['lightcoral' if 'Single' in m else 'steelblue'
                      for m in sorted_df['Model']],
               edgecolor='black')
for i, v in enumerate(sorted_df['F1']):
    ax.text(v + 0.005, i, f'{v:.4f}', va='center', fontweight='bold')
ax.set_xlabel('F1 Score')
ax.set_title('Ensemble Methods Comparison', fontweight='bold')
plt.tight_layout()
plt.show()

---
## 🔍 PART 2: MODEL EXPLAINABILITY (SHAP)
---

**WHY:** Models are black boxes. Stakeholders need to know WHY a prediction was made.

**SHAP (SHapley Additive exPlanations):**
- Based on game theory (Nobel Prize-winning concept!)
- Tells you how much EACH feature contributed to a prediction
- Works for ANY model

```python
# pip install shap
import shap

# For tree-based models (fast!)
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_test_s)

# Global importance (which features matter most overall)
shap.summary_plot(shap_values, X_test_s, feature_names=X.columns)

# Single prediction explanation
shap.force_plot(explainer.expected_value, shap_values[0], X_test_s.iloc[0])
```

**Interview Answer:**
"I use SHAP values to explain individual predictions. For example, 'This loan was flagged as high risk because credit score is 520 (contributing +25% to default probability) and debt-to-income ratio is 0.8 (+15%).' This makes the model transparent and builds trust with stakeholders."

In [ ]:
# ============================================================================
# FEATURE IMPORTANCE COMPARISON (Built-in)
# ============================================================================
print('🔍 FEATURE IMPORTANCE COMPARISON')
print('=' * 60)

# Get importance from tree-based models
importance_data = {}
for name, model in [('Random Forest', rf),
                     ('XGBoost', boosting_models['XGBoost']),
                     ('LightGBM', boosting_models['LightGBM'])]:
    if hasattr(model, 'feature_importances_'):
        imp = pd.Series(model.feature_importances_, index=X.columns)
        importance_data[name] = imp

# Plot top 10 features from each model
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
colors = ['darkorange', 'crimson', 'forestgreen']

for ax, (name, imp), color in zip(axes, importance_data.items(), colors):
    top10 = imp.sort_values(ascending=True).tail(10)
    ax.barh(top10.index, top10.values, color=color, edgecolor='black')
    ax.set_title(f'{name}', fontweight='bold')
    ax.set_xlabel('Importance')

plt.suptitle('Feature Importance Across Models', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('💡 Consistent top features across models = truly important features!')
print('   Features that vary = model-specific, less reliable.')

In [ ]:
# ============================================================================
# LEARNING CURVES
# ============================================================================
print('📈 LEARNING CURVES')
print('=' * 60)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (name, model) in zip(axes, [
    ('Random Forest', RandomForestClassifier(n_estimators=100, max_depth=10,
        class_weight='balanced', random_state=42, n_jobs=-1)),
    ('XGBoost', XGBClassifier(n_estimators=100, max_depth=5,
        random_state=42, use_label_encoder=False,
        eval_metric='logloss', verbosity=0))
]):
    train_sizes, train_scores, val_scores = learning_curve(
        model, X_bal, y_bal,
        train_sizes=np.linspace(0.1, 1.0, 8),
        cv=3, scoring='f1', n_jobs=-1
    )
    ax.plot(train_sizes, train_scores.mean(axis=1), 'o-',
            label='Train', color='blue')
    ax.plot(train_sizes, val_scores.mean(axis=1), 'o-',
            label='Validation', color='red')
    ax.fill_between(train_sizes,
        train_scores.mean(axis=1) - train_scores.std(axis=1),
        train_scores.mean(axis=1) + train_scores.std(axis=1),
        alpha=0.1, color='blue')
    ax.fill_between(train_sizes,
        val_scores.mean(axis=1) - val_scores.std(axis=1),
        val_scores.mean(axis=1) + val_scores.std(axis=1),
        alpha=0.1, color='red')
    ax.set_title(f'{name} Learning Curve', fontweight='bold')
    ax.set_xlabel('Training Size')
    ax.set_ylabel('F1 Score')
    ax.legend()

plt.tight_layout()
plt.show()

print('\n📖 How to Read Learning Curves:')
print('  ✅ Both curves high & close → Good model, enough data')
print('  ⚠️ Train high, Val low → Overfitting (simplify model)')
print('  ⚠️ Both low → Underfitting (add complexity/features)')
print('  💡 Gap closing with more data → Get more data!')
print('  💡 Gap NOT closing → Model has reached its limit')

---
## 📊 PART 3: ADVANCED INTERVIEW TOPICS
---

### A/B Testing in ML
- Deploy Model A and Model B simultaneously
- Route 50% traffic to each
- Compare real-world metrics (conversion, default rate)
- Keep the winner, retire the loser
- Like a clinical trial for models!

### Model Versioning
- Track: model version, training data version, hyperparameters, metrics
- Tools: MLflow, DVC, Weights & Biases
- Why: Reproducibility, rollback if new model fails

### Feature Store
- Centralized repository of features
- Ensures training features = serving features
- Prevents training-serving skew
- Tools: Feast, Tecton, Hopsworks

### MLOps
```
Data → Feature Engineering → Training → Evaluation → Deployment → Monitoring
  ↑                                                              |
  └──────────── Retraining Pipeline (automated) ←────────────────┘
```
- CI/CD for ML models
- Automated retraining triggers
- Model registry and governance

### Statistical Tests for ML
| Test | Use When |
|------|----------|
| Chi-Square | Two categorical variables |
| T-Test | Compare means of 2 groups |
| ANOVA | Compare means of 3+ groups |
| KS Test | Compare two distributions |
| Shapiro-Wilk | Check normality |

### Time Series Considerations
- NEVER use random train-test split
- Use TimeSeriesSplit for CV
- Beware look-ahead bias
- Feature engineering: lag features, rolling stats, seasonality
- Models: ARIMA, Prophet, LSTM, or XGBoost with time features

---
## 🏆 FINAL CHEAT SHEET
---

### Algorithm Selection
```
Target = NUMBER? → Linear/Ridge/Lasso/RF/XGB Regressor
Target = CATEGORY?
  Interpretable? → Logistic Regression / Decision Tree
  Best accuracy? → XGBoost / LightGBM
  Small data? → Naive Bayes
  High dimensions? → SVM
No labels? → KMeans / DBSCAN / PCA
```

### Metric Selection
```
Balanced → Accuracy
Imbalanced → F1, AUC
FP costly → Precision
FN costly → Recall
Regression → RMSE, MAE, R²
```

### Preprocessing Checklist
```
☐ Check missing values → Impute (median/mode)
☐ Check outliers → IQR cap
☐ Encode categoricals → Label/One-Hot
☐ Scale features → StandardScaler (if needed)
☐ Handle imbalance → SMOTE + class_weight
☐ Split BEFORE preprocessing → Prevent leakage
☐ Use Pipeline → Consistent & clean
```

### The 5 Questions That Matter
1. **WHAT** problem am I solving?
2. **WHAT** does my data look like?
3. **WHICH** tool fits best?
4. **HOW** do I measure success?
5. **HOW** do I explain it to others?

**Master these 5 → Ace any ML interview!** 🚀